In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("diabetes_dataset.csv")

df.head()

,EXERANY2,_BMI5CAT,_SMOKER3,MENTHLTH,SSBFRUT3,ALCDAY4,DIABETE4
0,2.0,3.0,4.0,2.0,888.0,888.0,3.0
1,1.0,3.0,4.0,88.0,888.0,888.0,3.0
2,2.0,3.0,2.0,2.0,101.0,101.0,3.0
3,2.0,4.0,4.0,88.0,888.0,230.0,3.0
4,1.0,4.0,4.0,25.0,301.0,888.0,3.0


In [3]:
df=df[df['DIABETE4'].isin([1,3])]

**Split Training and Test Set**

In [5]:
# Seperate training and test data
X, y = df.loc[:, ~df.columns.isin(['DIABETE4'])], df.loc[:, df.columns.isin(['DIABETE4'])]

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=1)

In [7]:
print('Training set size: {}'.format(len(X_train)))
print('Test set size: {}'.format(len(X_test)))

Training set size: 80416
Test set size: 20104


**Building a Naive Bayes Model**

In [9]:
def predict_logprob(X, W, W_prior):
    '''
    Computes the log probability of each class given input features.
    Input:
        - X: input features
        - W: model weights
        - W_prior: weight prior probabilities
    Output:
        - y_pred: probability of positive product review
    '''
    y_pred=None
    # Ensure features are positive for NB, negative likelihood doesnt make sense. 
    X = X - X.min()
    # Compute log probability using dot product of features 
    #   with log of class conditional probabilities
    y_pred = np.dot(X, np.log(W.T))
    # Add log prior probabilities of classes
    y_pred += np.log(W_prior)
    return y_pred

In [10]:
def predict_probability(X, W, W_prior, classes):
    '''
    Produces probabilistic estimate for P(y_i = +1 | x_i)
        Estimate ranges between 0 and 1.
    Input:
        - X: Input features
        - W: model weights
        - W_prior: weight prior probabilities
        - classes: class labels for input features
    Output:
        - y_pred: probability of positive product review
    '''
    y_pred=None
    # Ensure features are positive for NB
    X = X - X.min()
        
    if not isinstance(X, np.ndarray):
        X = np.array(X)
    # Identify index for positive class
    pos_class_idx = np.where(classes == 1)[0][0]
    # Get log probabilities
    y_pred = predict_logprob(X, W, W_prior)
    # Convert log probabilities to probabilities using the softmax function
    probs = np.exp(np.array(y_pred))
    probs = np.exp(probs) / np.sum(np.exp(probs),axis=1)[:, None]
    # Extract probability of positive class
    y_pred = probs[:, pos_class_idx]
    return y_pred

In [11]:
def predict(X, W, W_prior, classes):
    '''
    Hypothetical function  h(x)
    Input: 
        - X: Input features
        - W: model weights
        - W_prior: weight prior probabilities 
        - classes: class labels for input features
    Output:
        - y_pred: list of predicted classes 
    '''
    y_pred=None
    # Ensure features are positive for NB
    X = X - X.min()
    # Compute log probabilities
    y_pred = predict_logprob(X, W, W_prior)
    # Select class with highest probability
    y_pred = np.argmax(y_pred, axis=1)
    # Convert indices to class labels
    mapping = {i: k for i, k in enumerate(classes)}
    idx_to_class = np.vectorize(mapping.get)
    y_pred = idx_to_class(y_pred)
    return y_pred

In [12]:
def fit(X, Y, alpha=1):
    '''
    Compute closed-form solution for Naive Bayes classifier
    Input: 
        - X: Input features
        - Y: list of actual product sentiment classes 
        - alpha: laplace smoothing parameter
    Output:
        - W: predicted weights
        - W_prior: prior
        - likelihood_history: word likelihood (float)
    '''
    # Number of examples, Number of features
    num_examples, num_features = X.shape
    classes = np.unique(np.ravel(Y))
    num_classes = len(classes)
    # Initialization: weights, prior, likelihood
    W = np.zeros((num_classes, num_features))
    W_prior = np.zeros(num_classes)
    likelihood_history=[]

    # Compute class-conditional probabilities and class priors
    for ind, class_k in enumerate(classes):
        # Select samples belonging to class_k
        X_class_k = X[Y == class_k]
        # Compute likelihood using Laplace smoothing
        W[ind] = (np.sum(X_class_k, axis=0) + alpha)
        W[ind] /= (np.sum(X_class_k) + (alpha * X_class_k.shape[-1]))
        # Compute prior
        W_prior[ind] = X_class_k.shape[0] / num_examples
    
    # Compute and store log likelihood history
    log_likelihood = np.log(predict_probability(X, W, W_prior, classes)).mean()
    likelihood_history.append(log_likelihood)
    return W, W_prior, likelihood_history

In [13]:
model_weights, model_weights_prior, likelihood_history = fit(X_train, np.ravel(y_train))

/opt/anaconda3/lib/python3.12/site-packages/numpy/core/fromnumeric.py:86: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return reduction(axis=axis, out=out, **passkwargs)


In [14]:
print("Model weights:", model_weights)

Model weights: [[0.99968392 0.99987253 0.99987099 0.99999261 0.99999937 0.99999933]
 [0.99993877 0.9999753  0.99997884 0.9999987  0.99999987 0.99999986]]


In [15]:
print("Model weights prior:", model_weights_prior)

Model weights prior: [0.14480949 0.85519051]


In [16]:
print("Likelihood_history:", likelihood_history)

Likelihood_history: [-1.110116200789439]


**Classification Accuracy**

In [18]:
def get_classification_accuracy(prediction_labels, true_labels):    
    # Compute the number of correctly classified examples
    num_correct = np.sum(prediction_labels == true_labels)

    # Then compute accuracy by dividing num_correct by total number of examples
    accuracy = num_correct / len(true_labels)
    return accuracy

In [19]:
accuracy = get_classification_accuracy(predict(X_train, 
                                               model_weights, 
                                               model_weights_prior, 
                                               np.unique(np.ravel(y_train))),
                                       np.ravel(y_train))
print(accuracy)

0.8551905093513729


In [20]:
accuracy = get_classification_accuracy(predict(X_test, 
                                               model_weights, 
                                               model_weights_prior, 
                                               np.unique(np.ravel(y_test))),
                                       np.ravel(y_test))
print(accuracy)

0.857391563867887


**Classification Precision**

In [22]:
y_pred = predict(X_test, model_weights, model_weights_prior, np.unique(np.ravel(y_test)))
y_test = np.ravel(y_test)

In [23]:
np.unique(y_pred)

array([3.])

In [24]:
print(len(y_pred), len(y_test))

20104 20104


In [25]:
y_pred = np.array(y_pred).flatten()

true_pos = 0

for i in range(len(y_pred)):
    if y_test[i] == 1 and y_pred[i] == 1:
        true_pos += 1

print(true_pos)

0


In [26]:
false_pos = 0

for i in range(len(y_pred)):
    if y_test[i] == 3 and y_pred[i] == 1:
        false_pos += 1

print(false_pos)

0


In [27]:
precision = true_pos/(true_pos+false_pos)
print("Precision on test data: %s" % precision)

ZeroDivisionError: division by zero

**Recall**

In [29]:
false_neg = 0

for i in range(len(y_pred)):
    if y_test[i] == 1 and y_pred[i] == 3:
        false_neg += 1

print(false_neg)

2867


In [31]:
recall = true_pos / (true_pos + false_neg)
print("Recall on test data: %s" % recall)

Recall on test data: 0.0


In [33]:
print(np.unique(y_train))
print(np.unique(y_test))
print(np.unique(y_pred))

[1. 3.]
[1. 3.]
[3.]


In [35]:
from collections import Counter
print("y_train distribution:", Counter(y_train))
print("y_test distribution:", Counter(y_test))

y_train distribution: Counter({'DIABETE4': 1})
y_test distribution: Counter({3.0: 17237, 1.0: 2867})


In [37]:
print(model_weights_prior) #Class 3 prior much larger than class 1

[0.14480949 0.85519051]


**Undersample Majority Class**

In [40]:
# Convert to numpy
X_train = np.array(X_train)
y_train = np.array(y_train).flatten()

# Separate classes
X_majority = X_train[y_train == 3]
X_minority = X_train[y_train == 1]

y_majority = y_train[y_train == 3]
y_minority = y_train[y_train == 1]

# Get minority size
n_minority = len(X_minority)

# Randomly sample from majority WITHOUT replacement
indices = np.random.choice(len(X_majority), size=n_minority, replace=False)

X_majority_downsampled = X_majority[indices]
y_majority_downsampled = y_majority[indices]

# Combine
X_train_balanced = np.vstack((X_majority_downsampled, X_minority))
y_train_balanced = np.hstack((y_majority_downsampled, y_minority))

# Shuffle
shuffle_idx = np.random.permutation(len(y_train_balanced))
X_train_balanced = X_train_balanced[shuffle_idx]
y_train_balanced = y_train_balanced[shuffle_idx]

In [42]:
from collections import Counter
print(Counter(y_train_balanced))

Counter({1.0: 11645, 3.0: 11645})


In [48]:
model_weights, model_weights_prior, likelihood_history = fit(X_train_balanced, np.ravel(y_train_balanced))

In [50]:
print("Model weights prior:", model_weights_prior)

Model weights prior: [0.5 0.5]


In [52]:
accuracy = get_classification_accuracy(predict(X_train_balanced, 
                                               model_weights, 
                                               model_weights_prior, 
                                               np.unique(np.ravel(y_train_balanced))),
                                       np.ravel(y_train_balanced))
print(accuracy)

0.5801202232717905


In [54]:
accuracy = get_classification_accuracy(predict(X_test, 
                                               model_weights, 
                                               model_weights_prior, 
                                               np.unique(np.ravel(y_test))),
                                       np.ravel(y_test))
print(accuracy)

0.5245224830879427


In [56]:
y_pred = predict(X_test, model_weights, model_weights_prior, np.unique(np.ravel(y_test)))
y_test = np.ravel(y_test)

In [58]:
np.unique(y_pred)

array([1., 3.])

In [60]:
y_pred = np.array(y_pred).flatten()

true_pos = 0

for i in range(len(y_pred)):
    if y_test[i] == 1 and y_pred[i] == 1:
        true_pos += 1

print(true_pos)

1937


In [62]:
false_pos = 0

for i in range(len(y_pred)):
    if y_test[i] == 3 and y_pred[i] == 1:
        false_pos += 1

print(false_pos)

8629


In [64]:
precision = true_pos/(true_pos+false_pos)
print("Precision on test data: %s" % precision)

Precision on test data: 0.18332386901381792


In [66]:
false_neg = 0

for i in range(len(y_pred)):
    if y_test[i] == 1 and y_pred[i] == 3:
        false_neg += 1

print(false_neg)

930


In [68]:
recall = true_pos / (true_pos + false_neg)
print("Recall on test data: %s" % recall)

Recall on test data: 0.6756191140565051


In [70]:
if precision + recall == 0:
    f1 = 0.0
else:
    f1 = 2 * (precision * recall) / (precision + recall)

print("F1 score:", f1)

F1 score: 0.2883942529591305
